# SQL Server: Aggregate Functions, GROUP BY, HAVING, Extended Grouping and Joins

**Course:** Comprehensive Data Science (Data Scientist) · **Topic:** Query Writing, session notes
**Databases used:** `pubs`, `Northwind`, `AdventureWorksDW2022`

---

### How to read this notebook

* Every section follows the same pattern: **what it is → why we use it → example → output → key takeaway**.
* The code cells contain **T-SQL (SQL Server)**. The notebook runs on a Python kernel, so the cells are stored as reference code and each one carries its **result already saved as output**. You can read the notebook, push it to GitHub and see every result without a database connection.
* Outputs were produced from sample copies of the course databases. `pubs` is reproduced in full (titles, publishers, authors, sales). The `Northwind` and `AdventureWorksDW2022` tables are small illustrative samples, so counts and totals for those two will differ from your full databases, but the logic and the shape of every result are identical.
* `NULL` in an output means the value is SQL `NULL`.

### Contents

| # | Section | Main ideas |
|---|---------|-----------|
| 1 | Aggregate functions | `SUM`, `MAX`, `MIN`, `AVG`, `COUNT`, `ISNULL`, `DISTINCT` |
| 2 | `GROUP BY` | grouping columns and expressions, `DISTINCT` vs `GROUP BY` |
| 3 | Logical order of a query | why `WHERE` cannot see aliases or aggregates |
| 4 | `HAVING` vs `WHERE` | filtering rows vs filtering groups |
| 5 | Date arithmetic inside an aggregate | `DATEDIFF`, `GETDATE` |
| 6 | Extended grouping | `WITH CUBE`, `WITH ROLLUP`, `GROUPING SETS`, `GROUPING()` |
| 7 | Practice with Northwind and pubs | counting by country/city, conditional counting with `CASE` |
| 8 | Joins | horizontal vs vertical combining, `INNER JOIN`, multi-table joins |
| 9 | Useful helpers in join queries | `TOP ... WITH TIES`, `FORMAT`, `CONCAT_WS` |
| 10 | Join practice | pubs and Northwind exercises |
| 11 | Cheat sheet | one-page summary |

## 0. Function and keyword map

Everything used in this session, in one place.

| Function / keyword | What it does | Why we use it |
|---|---|---|
| `SUM(x)` | Adds up the non-`NULL` values | totals: revenue, quantity, hours |
| `MAX(x)` / `MIN(x)` | Largest / smallest non-`NULL` value (numbers, text, dates) | highest price, earliest date |
| `AVG(x)` | Mean of the non-`NULL` values | typical price, typical age |
| `COUNT(x)` | Number of non-`NULL` values in `x` | how many values exist |
| `COUNT(*)` / `COUNT(1)` | Number of rows | how many records exist |
| `COUNT(DISTINCT x)` | Number of unique non-`NULL` values | how many categories |
| `ISNULL(x, v)` | Replaces `NULL` with `v` | make `NULL` behave like 0 or a default label |
| `DISTINCT` | Removes duplicate rows from the result | list unique values |
| `GROUP BY` | Splits rows into groups, one result row per group | aggregate per category |
| `HAVING` | Filters groups after aggregation | "only groups whose average is above 15" |
| `WITH CUBE / WITH ROLLUP / GROUPING SETS` | Adds subtotal and grand-total rows | summary reports in one query |
| `GROUPING(col)` | 1 if the `NULL` in `col` is a subtotal marker, else 0 | tell subtotal `NULL`s from real `NULL`s |
| `YEAR(date)` | Extracts the year | group by year |
| `DATEDIFF(part, a, b)` | Number of `part` boundaries between two dates | ages, durations |
| `GETDATE()` | Current date and time | "as of today" calculations |
| `CASE WHEN ... THEN ... END` | Conditional value | conditional counting, pivot-style reports |
| `INNER JOIN ... ON` | Matches rows between tables | combine related tables |
| `TOP n [WITH TIES]` | Keeps the first `n` rows (plus ties) | top-N reports |
| `FORMAT(x, pattern)` | Formats a number or date as text | readable output |
| `CONCAT_WS(sep, a, b, ...)` | Joins strings with a separator, skipping `NULL`s | full names |

<a id="s1"></a>
## 1. Aggregate functions

### What they are
An **aggregate function** takes **many rows** and returns **one value**. The five you will use constantly are `SUM`, `MAX`, `MIN`, `AVG` and `COUNT`.

### Why we use them
Raw tables hold individual records. Analysis needs summaries: total sales, the most expensive book, the average salary, the number of customers. Aggregates compute those summaries inside the database, so only the answer travels to your Python or Power BI session instead of millions of rows.

### Two rules to remember from the start
1. Aggregates **ignore `NULL`** values (except `COUNT(*)`).
2. Without `GROUP BY`, an aggregate query returns **exactly one row** for the whole table.

First, look at the raw data we are about to summarize. The `price` column has two `NULL`s.

In [ ]:
use pubs
go
select t.price from dbo.titles as t;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>price</th></tr></thead><tbody><tr><td style="text-align:right">19.99</td></tr><tr><td style="text-align:right">11.95</td></tr><tr><td style="text-align:right">2.99</td></tr><tr><td style="text-align:right">19.99</td></tr><tr><td style="text-align:right">19.99</td></tr><tr><td style="text-align:right">2.99</td></tr><tr><td style="text-align:right"><i>NULL</i></td></tr><tr><td style="text-align:right">22.95</td></tr><tr><td style="text-align:right">20.00</td></tr><tr><td style="text-align:right"><i>NULL</i></td></tr><tr><td style="text-align:right">21.59</td></tr><tr><td style="text-align:right">10.95</td></tr><tr><td style="text-align:right">7.00</td></tr><tr><td style="text-align:right">19.99</td></tr><tr><td style="text-align:right">7.99</td></tr><tr><td style="text-align:right">20.95</td></tr><tr><td style="text-align:right">11.95</td></tr><tr><td style="text-align:right">14.99</td></tr></tbody></table>

(18 rows affected)

### 1.1 `SUM`

`SUM(column)` adds all non-`NULL` values.

In [ ]:
select SUM(t.price) as TotalPrice from dbo.titles as t;   -- 236.26

Warning: Null value is eliminated by an aggregate or other SET operation.

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>TotalPrice</th></tr></thead><tbody><tr><td style="text-align:right">236.26</td></tr></tbody></table>

(1 row affected)


SQL Server prints the warning *"Null value is eliminated by an aggregate..."* in the Messages tab. It is only telling you that `NULL`s were skipped.

#### `ISNULL`: turning `NULL` into 0

`ISNULL(x, 0)` replaces `NULL` with 0. For `SUM` the total does not change (adding 0 changes nothing), but it removes the warning and, more importantly, protects you from a group whose values are **all** `NULL`, where `SUM` would return `NULL` instead of 0.

In [ ]:
-- NULL => 0
select SUM(isnull(t.price, 0)) as TotalPrice from dbo.titles as t;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>TotalPrice</th></tr></thead><tbody><tr><td style="text-align:right">236.26</td></tr></tbody></table>

### 1.2 `MAX` and `MIN`

Largest and smallest value. They work on numbers, text (alphabetical order) and dates (latest / earliest).

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>MaxPrice</th></tr></thead><tbody><tr><td style="text-align:right">22.95</td></tr></tbody></table>

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>MinPrice</th></tr></thead><tbody><tr><td style="text-align:right">2.99</td></tr></tbody></table>

### 1.3 `COUNT` and its variants

`COUNT` has several forms that look similar but answer different questions.

| Form | Counts | `NULL` handling |
|---|---|---|
| `COUNT(*)` | all rows | counts rows even if every column is `NULL` |
| `COUNT(1)` | all rows | identical to `COUNT(*)` (the constant 1 is never `NULL`) |
| `COUNT(primary_key)` | all rows | same result, because a primary key can never be `NULL` |
| `COUNT(column)` | rows where `column` is not `NULL` | skips `NULL`s |
| `COUNT(DISTINCT column)` | unique non-`NULL` values | skips `NULL`s and duplicates |

In [ ]:
select count(price) as CountOfPrice      -- 16: two titles have no price
from dbo.titles as t;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>CountOfPrice</th></tr></thead><tbody><tr><td style="text-align:right">16</td></tr></tbody></table>

In [ ]:
-- Counting rows: four ways to get the same answer
select count(*)          as CountOfRows_Star,
       count(t.title_id) as CountOfRows_PK,     -- primary key is never NULL
       count(1)          as CountOfRows_One,
       count(t.price)    as CountOfPrice         -- different: NULL prices are skipped
from dbo.titles as t;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>CountOfRows_Star</th><th>CountOfRows_PK</th><th>CountOfRows_One</th><th>CountOfPrice</th></tr></thead><tbody><tr><td style="text-align:right">18</td><td style="text-align:right">18</td><td style="text-align:right">18</td><td style="text-align:right">16</td></tr></tbody></table>

`select *, 1 from ...` shows why `COUNT(1)` equals `COUNT(*)`: the literal `1` is attached to every row and is never `NULL`. (Only the first 3 rows are shown here; the lecture displays all 18.)

In [ ]:
select top 3 *, 1 as One from dbo.titles as t;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>title_id</th><th>title</th><th>type</th><th>pub_id</th><th>price</th><th>pubdate</th><th>One</th></tr></thead><tbody><tr><td>BU1032</td><td>The Busy Executive&#x27;s Database Guide</td><td>business</td><td>1389</td><td style="text-align:right">19.99</td><td>1991-06-12</td><td style="text-align:right">1</td></tr><tr><td>BU1111</td><td>Cooking with Computers: Surreptitious Balance Sheets</td><td>business</td><td>1389</td><td style="text-align:right">11.95</td><td>1991-06-09</td><td style="text-align:right">1</td></tr><tr><td>BU2075</td><td>You Can Combat Computer Stress!</td><td>business</td><td>0736</td><td style="text-align:right">2.99</td><td>1991-06-30</td><td style="text-align:right">1</td></tr></tbody></table>

#### `COUNT(DISTINCT ...)`: unique values

`DISTINCT` inside an aggregate removes duplicates **before** counting.

In [ ]:
-- how many different book types exist?
select count(distinct t.[type]) as CountOfType
from dbo.titles as t;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>CountOfType</th></tr></thead><tbody><tr><td style="text-align:right">6</td></tr></tbody></table>

### 1.4 `AVG`

`AVG(column)` = sum of non-`NULL` values ÷ **count of non-`NULL` values**. Because `NULL`s are excluded from the denominator too, the result can differ from what you expect.

In [ ]:
select AVG(t.price) as AveragePrice      -- 14.7662  (236.26 / 16)
from dbo.titles as t;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>AveragePrice</th></tr></thead><tbody><tr><td style="text-align:right">14.7662</td></tr></tbody></table>

In [ ]:
-- The effect of NULL on AVG: 16 prices vs 18 rows
select
    AVG(t.price)             as Avg_IgnoringNULL,     -- 236.26 / 16
    AVG(isnull(t.price, 0))  as Avg_NULLasZero        -- 236.26 / 18
from dbo.titles as t;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>Avg_IgnoringNULL</th><th>Avg_NULLasZero</th></tr></thead><tbody><tr><td style="text-align:right">14.7662</td><td style="text-align:right">13.1255</td></tr></tbody></table>

> **Takeaway:** choose deliberately. If a missing price means *unknown*, leave it as `NULL` and `AVG` skips it. If a missing price really means *zero*, use `ISNULL(price, 0)`.
> Also note: `AVG` of an **integer** column returns an **integer** in SQL Server (the decimals are cut off). Multiply by `1.0` or `CAST` to `DECIMAL` if you need decimals.

### 1.5 All five aggregates together

In [ ]:
select
    SUM(t.price)   as TotalPrice,
    MAX(t.price)   as MaxPrice,
    MIN(t.price)   as MinPrice,
    AVG(t.price)   as AveragePrice,
    COUNT(t.price) as CountOfPrice
from dbo.titles as t;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>TotalPrice</th><th>MaxPrice</th><th>MinPrice</th><th>AveragePrice</th><th>CountOfPrice</th></tr></thead><tbody><tr><td style="text-align:right">236.26</td><td style="text-align:right">22.95</td><td style="text-align:right">2.99</td><td style="text-align:right">14.7662</td><td style="text-align:right">16</td></tr></tbody></table>

### 1.6 A common error: mixing aggregated and non-aggregated columns

Once you use an aggregate, **every other column in `SELECT` must be inside an aggregate or listed in `GROUP BY`**. The database cannot show one `type` value next to an average that covers the whole table.

In [ ]:
select
    t.[type],                   -- non-aggregated column
    avg(price) as AVGPrice      -- aggregated column
from dbo.titles as t;

Msg 8120, Level 16, State 1
Column 'dbo.titles.type' is invalid in the select list because it is not contained in either an aggregate function or the GROUP BY clause.

<a id="s2"></a>
## 2. `GROUP BY`

### What it is
`GROUP BY` splits the rows into groups that share the same value(s) in the listed column(s), then applies the aggregate to **each group separately**. You get **one output row per group**.

### Why we use it
Almost every business question has "per": average price **per type**, sales **per publisher**, customers **per country**. `GROUP BY` is how you ask it.

### The complete clause order (as written)
```
SELECT
DISTINCT
TOP N
INTO          -- make-table query (creates a new table from the result)
FROM
WHERE
GROUP BY
HAVING
ORDER BY
```

### 2.1 Group by one column

In [ ]:
select
    t.[type],
    avg(price) as AVGPrice
from dbo.titles as t
group by t.[type];

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>type</th><th>AVGPrice</th></tr></thead><tbody><tr><td>UNDECIDED</td><td style="text-align:right"><i>NULL</i></td></tr><tr><td>business</td><td style="text-align:right">13.73</td></tr><tr><td>mod_cook</td><td style="text-align:right">11.49</td></tr><tr><td>popular_comp</td><td style="text-align:right">21.475</td></tr><tr><td>psychology</td><td style="text-align:right">13.504</td></tr><tr><td>trad_cook</td><td style="text-align:right">15.9633</td></tr></tbody></table>

Notice the `UNDECIDED` group: its only book has a `NULL` price, so its average is `NULL`. Also, rows whose grouping value is `NULL` would form **one** group of their own.

### 2.2 Group by an expression

You can group by a calculation such as `YEAR(date)`. The expression in `SELECT` must match the one in `GROUP BY`.

In [ ]:
select
    year(t.pubdate) as YearOfPub,
    avg(price)      as AVGPrice
from dbo.titles as t
group by year(t.pubdate);

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>YearOfPub</th><th>AVGPrice</th></tr></thead><tbody><tr><td style="text-align:right">1991</td><td style="text-align:right">14.4173</td></tr><tr><td style="text-align:right">1994</td><td style="text-align:right">20.00</td></tr></tbody></table>

### 2.3 `DISTINCT` vs `GROUP BY`

When you only want the list of unique values and no aggregate, both give the same result. `DISTINCT` is shorter and states the intent more clearly. Use `GROUP BY` when you also need an aggregate.

In [ ]:
-- with GROUP BY
select t.[type]
from dbo.titles as t
group by t.[type];

-- with DISTINCT
select distinct t.[type]
from dbo.titles as t;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>type</th></tr></thead><tbody><tr><td>UNDECIDED</td></tr><tr><td>business</td></tr><tr><td>mod_cook</td></tr><tr><td>popular_comp</td></tr><tr><td>psychology</td></tr><tr><td>trad_cook</td></tr></tbody></table>

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>type</th></tr></thead><tbody><tr><td>UNDECIDED</td></tr><tr><td>business</td></tr><tr><td>mod_cook</td></tr><tr><td>popular_comp</td></tr><tr><td>psychology</td></tr><tr><td>trad_cook</td></tr></tbody></table>

### 2.4 Group by several columns

Each **combination** of the listed columns becomes one group.

In [ ]:
select
    t.[type],
    year(t.pubdate) as YearOfPub,
    max(price)      as MaxOfPrice,
    min(price)      as MinOfPrice,
    avg(price)      as AVGOfPrice
from dbo.titles as t
group by t.[type], year(t.pubdate);

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>type</th><th>YearOfPub</th><th>MaxOfPrice</th><th>MinOfPrice</th><th>AVGOfPrice</th></tr></thead><tbody><tr><td>UNDECIDED</td><td style="text-align:right">1991</td><td style="text-align:right"><i>NULL</i></td><td style="text-align:right"><i>NULL</i></td><td style="text-align:right"><i>NULL</i></td></tr><tr><td>business</td><td style="text-align:right">1991</td><td style="text-align:right">19.99</td><td style="text-align:right">2.99</td><td style="text-align:right">13.73</td></tr><tr><td>mod_cook</td><td style="text-align:right">1991</td><td style="text-align:right">19.99</td><td style="text-align:right">2.99</td><td style="text-align:right">11.49</td></tr><tr><td>popular_comp</td><td style="text-align:right">1991</td><td style="text-align:right">22.95</td><td style="text-align:right">22.95</td><td style="text-align:right">22.95</td></tr><tr><td>popular_comp</td><td style="text-align:right">1994</td><td style="text-align:right">20.00</td><td style="text-align:right">20.00</td><td style="text-align:right">20.00</td></tr><tr><td>psychology</td><td style="text-align:right">1991</td><td style="text-align:right">21.59</td><td style="text-align:right">7.00</td><td style="text-align:right">13.504</td></tr><tr><td>trad_cook</td><td style="text-align:right">1991</td><td style="text-align:right">20.95</td><td style="text-align:right">11.95</td><td style="text-align:right">15.9633</td></tr></tbody></table>

### 2.5 Grouping by a unique column is pointless

`title` is unique, so every group contains exactly one row and `MAX(price)` is just the price itself. Group by a column only when many rows share its values.

In [ ]:
select
    t.title,                       -- unique
    max(price) as MaxPrice
from dbo.titles as t
group by t.title;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>title</th><th>MaxPrice</th></tr></thead><tbody><tr><td>The Busy Executive&#x27;s Database Guide</td><td style="text-align:right">19.99</td></tr><tr><td>Cooking with Computers: Surreptitious Balance Sheets</td><td style="text-align:right">11.95</td></tr><tr><td>You Can Combat Computer Stress!</td><td style="text-align:right">2.99</td></tr><tr><td>Straight Talk About Computers</td><td style="text-align:right">19.99</td></tr><tr><td>Silicon Valley Gastronomic Treats</td><td style="text-align:right">19.99</td></tr><tr><td>The Gourmet Microwave</td><td style="text-align:right">2.99</td></tr><tr><td>The Psychology of Computer Cooking</td><td style="text-align:right"><i>NULL</i></td></tr><tr><td>But Is It User Friendly?</td><td style="text-align:right">22.95</td></tr><tr><td>Secrets of Silicon Valley</td><td style="text-align:right">20.00</td></tr><tr><td>Net Etiquette</td><td style="text-align:right"><i>NULL</i></td></tr><tr><td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td><td style="text-align:right">21.59</td></tr><tr><td>Is Anger the Enemy?</td><td style="text-align:right">10.95</td></tr><tr><td>Life Without Fear</td><td style="text-align:right">7.00</td></tr><tr><td>Prolonged Data Deprivation: Four Case Studies</td><td style="text-align:right">19.99</td></tr><tr><td>Emotional Security: A New Algorithm</td><td style="text-align:right">7.99</td></tr><tr><td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td><td style="text-align:right">20.95</td></tr><tr><td>Fifty Years in Buckingham Palace Kitchens</td><td style="text-align:right">11.95</td></tr><tr><td>Sushi, Anyone?</td><td style="text-align:right">14.99</td></tr></tbody></table>

In [ ]:
select count(*) as CountOfRows from dbo.titles;               -- 18
select count(distinct t.title) as CountOfUniqueTitles from dbo.titles as t;   -- 18: every title is unique

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>CountOfRows</th></tr></thead><tbody><tr><td style="text-align:right">18</td></tr></tbody></table>

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>CountOfUniqueTitles</th></tr></thead><tbody><tr><td style="text-align:right">18</td></tr></tbody></table>

<a id="s3"></a>
## 3. Written order vs logical order

### Why this matters
You **write** `SELECT` first, but SQL Server **evaluates** the query in a different order. Most "why does this error?" questions are answered by this list.

| Step | Logical processing order | What happens |
|---|---|---|
| 1 | `FROM` / `JOIN` / `ON` | pick the tables, combine them |
| 2 | `WHERE` | filter individual rows |
| 3 | `GROUP BY` | form groups |
| 4 | `WITH CUBE / ROLLUP` | add subtotal rows |
| 5 | `HAVING` | filter groups |
| 6 | `SELECT` | compute the output columns and **create aliases** |
| 7 | `DISTINCT` | remove duplicate result rows |
| 8 | `ORDER BY` | sort (this is why it **can** use aliases) |
| 9 | `TOP` | keep the first N rows |

Consequences:
* `WHERE` runs before `GROUP BY`, so it **cannot use aggregates**.
* `WHERE` runs before `SELECT`, so it **cannot use column aliases**.
* `ORDER BY` runs after `SELECT`, so it **can** use aliases.

In [ ]:
-- WHERE cannot see an alias that SELECT has not created yet
select t.title, t.price * 2 as DoublePrice
from dbo.titles as t
where DoublePrice > 40;

Msg 207, Level 16, State 1
Invalid column name 'DoublePrice'.

In [ ]:
-- Fix 1: repeat the expression in WHERE
select t.title, t.price * 2 as DoublePrice
from dbo.titles as t
where t.price * 2 > 40
order by DoublePrice desc;          -- ORDER BY can use the alias

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>title</th><th>DoublePrice</th></tr></thead><tbody><tr><td>But Is It User Friendly?</td><td style="text-align:right">45.90</td></tr><tr><td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td><td style="text-align:right">43.18</td></tr><tr><td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td><td style="text-align:right">41.90</td></tr></tbody></table>

<a id="s4"></a>
## 4. `HAVING` vs `WHERE`

### What they are
* `WHERE` filters **rows**, *before* grouping.
* `HAVING` filters **groups**, *after* aggregation, so it can use aggregates.

### Why we need both
"Average price per type, only for books not published in 2026, and only types whose average exceeds 15." The first condition is about rows; the second is about a group result. They belong in different clauses.

First the mistake: an aggregate inside `WHERE`.

In [ ]:
select
    t.[type],
    avg(price) as avgPrice
from dbo.titles as t
where year(t.pubdate) <> 2026 and avg(price) > 15       -- ERROR
group by t.[type]
order by avgPrice desc;

Msg 147, Level 16, State 1
An aggregate may not appear in the WHERE clause unless it is in a subquery contained in a HAVING clause or a select list, and the column being aggregated is an outer reference.

The correct version: row filter in `WHERE`, group filter in `HAVING`.

> **Performance tip:** always filter rows in `WHERE` when you can. It shrinks the data **before** grouping, so there is less work to aggregate. Use `HAVING` only for conditions on aggregates.

In [ ]:
select
    t.[type],
    avg(price) as avgPrice
from dbo.titles as t
where year(t.pubdate) <> 2026       -- row filter (before grouping)
group by t.[type]
having avg(price) > 15              -- group filter (after grouping)
order by avgPrice desc;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>type</th><th>avgPrice</th></tr></thead><tbody><tr><td>popular_comp</td><td style="text-align:right">21.475</td></tr><tr><td>trad_cook</td><td style="text-align:right">15.9633</td></tr></tbody></table>

Only two types survive. Without `HAVING` you would see all six groups.

<a id="s5"></a>
## 5. Dates inside aggregates: `DATEDIFF` and `GETDATE`

* `GETDATE()` returns the current date and time of the server.
* `DATEDIFF(part, start, end)` counts how many `part` boundaries are crossed between two dates. With `year` it is the difference of the calendar years, **not** the number of completed years. Someone born on 1990-12-31 is "1 year old" on 1991-01-01. For a rough average age this is acceptable; for exact ages you need extra logic.

Here we compute the **average age of customers** by education level and marital status. Output computed on 2026-10-11, so ages move whenever you re-run it on another day.

In [ ]:
use AdventureWorksDW2022
go
select
    dc.EnglishEducation,
    dc.MaritalStatus,
    avg(datediff(year, dc.Birthdate, getdate())) as AVG_Age
from dbo.DimCustomer as dc
group by
    dc.EnglishEducation,
    dc.MaritalStatus
order by EnglishEducation asc;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>EnglishEducation</th><th>MaritalStatus</th><th>AVG_Age</th></tr></thead><tbody><tr><td>Bachelors</td><td>M</td><td style="text-align:right">50</td></tr><tr><td>Bachelors</td><td>S</td><td style="text-align:right">38</td></tr><tr><td>Graduate Degree</td><td>M</td><td style="text-align:right">60</td></tr><tr><td>Graduate Degree</td><td>S</td><td style="text-align:right">41</td></tr><tr><td>High School</td><td>M</td><td style="text-align:right">64</td></tr><tr><td>High School</td><td>S</td><td style="text-align:right">51</td></tr><tr><td>Partial College</td><td>M</td><td style="text-align:right">57</td></tr><tr><td>Partial College</td><td>S</td><td style="text-align:right">38</td></tr><tr><td>Partial High School</td><td>S</td><td style="text-align:right">31</td></tr></tbody></table>

> `DATEDIFF` returns an `int`, so `AVG` returns an `int` too (truncated, not rounded).

<a id="s6"></a>
## 6. Extended grouping: `ROLLUP`, `CUBE`, `GROUPING SETS`

### What they are
A normal `GROUP BY a, b, c` returns only the most detailed level. Management reports usually also want **subtotals and a grand total**. These three options produce them in a single query instead of several queries glued together with `UNION`.

| Option | Which groupings it produces | Number of groupings for 3 columns |
|---|---|---|
| plain `GROUP BY a,b,c` | only `(a,b,c)` | 1 |
| `WITH ROLLUP` | hierarchical: `(a,b,c)`, `(a,b)`, `(a)`, `()` | 4 |
| `WITH CUBE` | **every** combination of the columns | 8 |
| `GROUPING SETS (...)` | **exactly** the combinations you list | your choice |

### How to read the output
In a subtotal row the column that was rolled up shows **`NULL`**. A row with `NULL` in every grouping column is the **grand total** `()`.

Baseline first: the grand total alone, and the plain three-column grouping.

In [ ]:
use AdventureWorksDW2022
go
-- Grand total only (no GROUP BY)
select
    count(1)              as CountOfEmployee,
    avg(de.VacationHours) as AVGVacationHours
from dbo.DimEmployee as de;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>CountOfEmployee</th><th>AVGVacationHours</th></tr></thead><tbody><tr><td style="text-align:right">12</td><td style="text-align:right">33</td></tr></tbody></table>

In [ ]:
-- Plain GROUP BY: only the most detailed level
select
    de.DepartmentName,
    de.Gender,
    de.MaritalStatus,
    count(1)              as CountOfEmployee,
    avg(de.VacationHours) as AVGVacationHours
from dbo.DimEmployee as de
group by
    de.DepartmentName,
    de.MaritalStatus,
    de.Gender
order by DepartmentName;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>DepartmentName</th><th>Gender</th><th>MaritalStatus</th><th>CountOfEmployee</th><th>AVGVacationHours</th></tr></thead><tbody><tr><td>Engineering</td><td>F</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">22</td></tr><tr><td>Engineering</td><td>M</td><td>M</td><td style="text-align:right">2</td><td style="text-align:right">21</td></tr><tr><td>Engineering</td><td>M</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">18</td></tr><tr><td>Marketing</td><td>F</td><td>M</td><td style="text-align:right">1</td><td style="text-align:right">35</td></tr><tr><td>Marketing</td><td>F</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">41</td></tr><tr><td>Marketing</td><td>M</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">27</td></tr><tr><td>Sales</td><td>F</td><td>M</td><td style="text-align:right">1</td><td style="text-align:right">44</td></tr><tr><td>Sales</td><td>F</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">39</td></tr><tr><td>Sales</td><td>M</td><td>M</td><td style="text-align:right">2</td><td style="text-align:right">49</td></tr><tr><td>Sales</td><td>M</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">33</td></tr></tbody></table>

### 6.1 `WITH ROLLUP`: hierarchical subtotals

The column order defines the hierarchy: here Department → MaritalStatus → Gender. You get the detail rows, then a subtotal for each (Department, MaritalStatus), then for each Department, then the grand total.

In [ ]:
select
    de.DepartmentName,
    de.Gender,
    de.MaritalStatus,
    count(1)              as CountOfEmployee,
    avg(de.VacationHours) as AVGVacationHours
from dbo.DimEmployee as de
group by
    de.DepartmentName,
    de.MaritalStatus,
    de.Gender with rollup
order by DepartmentName;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>DepartmentName</th><th>Gender</th><th>MaritalStatus</th><th>CountOfEmployee</th><th>AVGVacationHours</th></tr></thead><tbody><tr><td><i>NULL</i></td><td><i>NULL</i></td><td><i>NULL</i></td><td style="text-align:right">12</td><td style="text-align:right">33</td></tr><tr><td>Engineering</td><td><i>NULL</i></td><td><i>NULL</i></td><td style="text-align:right">4</td><td style="text-align:right">20</td></tr><tr><td>Engineering</td><td><i>NULL</i></td><td>M</td><td style="text-align:right">2</td><td style="text-align:right">21</td></tr><tr><td>Engineering</td><td>M</td><td>M</td><td style="text-align:right">2</td><td style="text-align:right">21</td></tr><tr><td>Engineering</td><td><i>NULL</i></td><td>S</td><td style="text-align:right">2</td><td style="text-align:right">20</td></tr><tr><td>Engineering</td><td>F</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">22</td></tr><tr><td>Engineering</td><td>M</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">18</td></tr><tr><td>Marketing</td><td><i>NULL</i></td><td><i>NULL</i></td><td style="text-align:right">3</td><td style="text-align:right">34</td></tr><tr><td>Marketing</td><td><i>NULL</i></td><td>M</td><td style="text-align:right">1</td><td style="text-align:right">35</td></tr><tr><td>Marketing</td><td>F</td><td>M</td><td style="text-align:right">1</td><td style="text-align:right">35</td></tr><tr><td>Marketing</td><td><i>NULL</i></td><td>S</td><td style="text-align:right">2</td><td style="text-align:right">34</td></tr><tr><td>Marketing</td><td>F</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">41</td></tr><tr><td>Marketing</td><td>M</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">27</td></tr><tr><td>Sales</td><td><i>NULL</i></td><td><i>NULL</i></td><td style="text-align:right">5</td><td style="text-align:right">42</td></tr><tr><td>Sales</td><td><i>NULL</i></td><td>M</td><td style="text-align:right">3</td><td style="text-align:right">47</td></tr><tr><td>Sales</td><td>F</td><td>M</td><td style="text-align:right">1</td><td style="text-align:right">44</td></tr><tr><td>Sales</td><td>M</td><td>M</td><td style="text-align:right">2</td><td style="text-align:right">49</td></tr><tr><td>Sales</td><td><i>NULL</i></td><td>S</td><td style="text-align:right">2</td><td style="text-align:right">36</td></tr><tr><td>Sales</td><td>F</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">39</td></tr><tr><td>Sales</td><td>M</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">33</td></tr></tbody></table>

### 6.2 `WITH CUBE`: every combination

`CUBE` adds the subtotals for **all** combinations, including ones `ROLLUP` skips, such as "by Gender only" or "by MaritalStatus only", across all departments.

In [ ]:
select
    de.DepartmentName,
    de.Gender,
    de.MaritalStatus,
    count(1)              as CountOfEmployee,
    avg(de.VacationHours) as AVGVacationHours
from dbo.DimEmployee as de
group by
    de.DepartmentName,
    de.MaritalStatus,
    de.Gender with cube
order by DepartmentName;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>DepartmentName</th><th>Gender</th><th>MaritalStatus</th><th>CountOfEmployee</th><th>AVGVacationHours</th></tr></thead><tbody><tr><td><i>NULL</i></td><td><i>NULL</i></td><td><i>NULL</i></td><td style="text-align:right">12</td><td style="text-align:right">33</td></tr><tr><td><i>NULL</i></td><td><i>NULL</i></td><td>M</td><td style="text-align:right">6</td><td style="text-align:right">36</td></tr><tr><td><i>NULL</i></td><td><i>NULL</i></td><td>S</td><td style="text-align:right">6</td><td style="text-align:right">30</td></tr><tr><td><i>NULL</i></td><td>F</td><td><i>NULL</i></td><td style="text-align:right">5</td><td style="text-align:right">36</td></tr><tr><td><i>NULL</i></td><td>F</td><td>M</td><td style="text-align:right">2</td><td style="text-align:right">39</td></tr><tr><td><i>NULL</i></td><td>F</td><td>S</td><td style="text-align:right">3</td><td style="text-align:right">34</td></tr><tr><td><i>NULL</i></td><td>M</td><td><i>NULL</i></td><td style="text-align:right">7</td><td style="text-align:right">31</td></tr><tr><td><i>NULL</i></td><td>M</td><td>M</td><td style="text-align:right">4</td><td style="text-align:right">35</td></tr><tr><td><i>NULL</i></td><td>M</td><td>S</td><td style="text-align:right">3</td><td style="text-align:right">26</td></tr><tr><td>Engineering</td><td><i>NULL</i></td><td><i>NULL</i></td><td style="text-align:right">4</td><td style="text-align:right">20</td></tr><tr><td>Engineering</td><td><i>NULL</i></td><td>M</td><td style="text-align:right">2</td><td style="text-align:right">21</td></tr><tr><td>Engineering</td><td><i>NULL</i></td><td>S</td><td style="text-align:right">2</td><td style="text-align:right">20</td></tr><tr><td>Engineering</td><td>F</td><td><i>NULL</i></td><td style="text-align:right">1</td><td style="text-align:right">22</td></tr><tr><td>Engineering</td><td>F</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">22</td></tr><tr><td>Engineering</td><td>M</td><td><i>NULL</i></td><td style="text-align:right">3</td><td style="text-align:right">20</td></tr><tr><td>Engineering</td><td>M</td><td>M</td><td style="text-align:right">2</td><td style="text-align:right">21</td></tr><tr><td>Engineering</td><td>M</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">18</td></tr><tr><td>Marketing</td><td><i>NULL</i></td><td><i>NULL</i></td><td style="text-align:right">3</td><td style="text-align:right">34</td></tr><tr><td>Marketing</td><td><i>NULL</i></td><td>M</td><td style="text-align:right">1</td><td style="text-align:right">35</td></tr><tr><td>Marketing</td><td><i>NULL</i></td><td>S</td><td style="text-align:right">2</td><td style="text-align:right">34</td></tr><tr><td>Marketing</td><td>F</td><td><i>NULL</i></td><td style="text-align:right">2</td><td style="text-align:right">38</td></tr><tr><td>Marketing</td><td>F</td><td>M</td><td style="text-align:right">1</td><td style="text-align:right">35</td></tr><tr><td>Marketing</td><td>F</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">41</td></tr><tr><td>Marketing</td><td>M</td><td><i>NULL</i></td><td style="text-align:right">1</td><td style="text-align:right">27</td></tr><tr><td>Marketing</td><td>M</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">27</td></tr><tr><td>Sales</td><td><i>NULL</i></td><td><i>NULL</i></td><td style="text-align:right">5</td><td style="text-align:right">42</td></tr><tr><td>Sales</td><td><i>NULL</i></td><td>M</td><td style="text-align:right">3</td><td style="text-align:right">47</td></tr><tr><td>Sales</td><td><i>NULL</i></td><td>S</td><td style="text-align:right">2</td><td style="text-align:right">36</td></tr><tr><td>Sales</td><td>F</td><td><i>NULL</i></td><td style="text-align:right">2</td><td style="text-align:right">41</td></tr><tr><td>Sales</td><td>F</td><td>M</td><td style="text-align:right">1</td><td style="text-align:right">44</td></tr><tr><td>Sales</td><td>F</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">39</td></tr><tr><td>Sales</td><td>M</td><td><i>NULL</i></td><td style="text-align:right">3</td><td style="text-align:right">43</td></tr><tr><td>Sales</td><td>M</td><td>M</td><td style="text-align:right">2</td><td style="text-align:right">49</td></tr><tr><td>Sales</td><td>M</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">33</td></tr></tbody></table>

### 6.3 `GROUPING SETS`: choose the groupings yourself

`CUBE` can produce more rows than you need. `GROUPING SETS` lets you list exactly which groupings you want; `()` means the grand total.

In [ ]:
select
    de.DepartmentName,
    de.Gender,
    de.MaritalStatus,
    count(1)              as CountOfEmployee,
    avg(de.VacationHours) as AVGVacationHours
from dbo.DimEmployee as de
group by grouping sets (
        (de.DepartmentName, de.Gender, de.MaritalStatus),
        (),                                  -- grand total
        (de.Gender),
        (de.MaritalStatus),
        (de.Gender, de.MaritalStatus),
        (de.Gender, de.DepartmentName),
        (de.MaritalStatus, de.DepartmentName)
)
order by DepartmentName;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>DepartmentName</th><th>Gender</th><th>MaritalStatus</th><th>CountOfEmployee</th><th>AVGVacationHours</th></tr></thead><tbody><tr><td><i>NULL</i></td><td><i>NULL</i></td><td><i>NULL</i></td><td style="text-align:right">12</td><td style="text-align:right">33</td></tr><tr><td><i>NULL</i></td><td><i>NULL</i></td><td>M</td><td style="text-align:right">6</td><td style="text-align:right">36</td></tr><tr><td><i>NULL</i></td><td><i>NULL</i></td><td>S</td><td style="text-align:right">6</td><td style="text-align:right">30</td></tr><tr><td><i>NULL</i></td><td>F</td><td><i>NULL</i></td><td style="text-align:right">5</td><td style="text-align:right">36</td></tr><tr><td><i>NULL</i></td><td>F</td><td>M</td><td style="text-align:right">2</td><td style="text-align:right">39</td></tr><tr><td><i>NULL</i></td><td>F</td><td>S</td><td style="text-align:right">3</td><td style="text-align:right">34</td></tr><tr><td><i>NULL</i></td><td>M</td><td><i>NULL</i></td><td style="text-align:right">7</td><td style="text-align:right">31</td></tr><tr><td><i>NULL</i></td><td>M</td><td>M</td><td style="text-align:right">4</td><td style="text-align:right">35</td></tr><tr><td><i>NULL</i></td><td>M</td><td>S</td><td style="text-align:right">3</td><td style="text-align:right">26</td></tr><tr><td>Engineering</td><td><i>NULL</i></td><td>M</td><td style="text-align:right">2</td><td style="text-align:right">21</td></tr><tr><td>Engineering</td><td><i>NULL</i></td><td>S</td><td style="text-align:right">2</td><td style="text-align:right">20</td></tr><tr><td>Engineering</td><td>F</td><td><i>NULL</i></td><td style="text-align:right">1</td><td style="text-align:right">22</td></tr><tr><td>Engineering</td><td>F</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">22</td></tr><tr><td>Engineering</td><td>M</td><td><i>NULL</i></td><td style="text-align:right">3</td><td style="text-align:right">20</td></tr><tr><td>Engineering</td><td>M</td><td>M</td><td style="text-align:right">2</td><td style="text-align:right">21</td></tr><tr><td>Engineering</td><td>M</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">18</td></tr><tr><td>Marketing</td><td><i>NULL</i></td><td>M</td><td style="text-align:right">1</td><td style="text-align:right">35</td></tr><tr><td>Marketing</td><td><i>NULL</i></td><td>S</td><td style="text-align:right">2</td><td style="text-align:right">34</td></tr><tr><td>Marketing</td><td>F</td><td><i>NULL</i></td><td style="text-align:right">2</td><td style="text-align:right">38</td></tr><tr><td>Marketing</td><td>F</td><td>M</td><td style="text-align:right">1</td><td style="text-align:right">35</td></tr><tr><td>Marketing</td><td>F</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">41</td></tr><tr><td>Marketing</td><td>M</td><td><i>NULL</i></td><td style="text-align:right">1</td><td style="text-align:right">27</td></tr><tr><td>Marketing</td><td>M</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">27</td></tr><tr><td>Sales</td><td><i>NULL</i></td><td>M</td><td style="text-align:right">3</td><td style="text-align:right">47</td></tr><tr><td>Sales</td><td><i>NULL</i></td><td>S</td><td style="text-align:right">2</td><td style="text-align:right">36</td></tr><tr><td>Sales</td><td>F</td><td><i>NULL</i></td><td style="text-align:right">2</td><td style="text-align:right">41</td></tr><tr><td>Sales</td><td>F</td><td>M</td><td style="text-align:right">1</td><td style="text-align:right">44</td></tr><tr><td>Sales</td><td>F</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">39</td></tr><tr><td>Sales</td><td>M</td><td><i>NULL</i></td><td style="text-align:right">3</td><td style="text-align:right">43</td></tr><tr><td>Sales</td><td>M</td><td>M</td><td style="text-align:right">2</td><td style="text-align:right">49</td></tr><tr><td>Sales</td><td>M</td><td>S</td><td style="text-align:right">1</td><td style="text-align:right">33</td></tr></tbody></table>

### 6.4 `GROUPING()`: is this `NULL` a subtotal or real data?

If a column can itself contain `NULL`, a `NULL` in a subtotal row is ambiguous. `GROUPING(column)` returns **1** when the `NULL` was produced by the rollup and **0** when it is a real value. It is also the clean way to put readable labels such as *"All departments"* into the report.

In [ ]:
select
    case when grouping(de.DepartmentName) = 1 then 'All departments' else de.DepartmentName end as Department,
    case when grouping(de.Gender)         = 1 then 'All genders'     else de.Gender         end as Gender,
    count(1) as CountOfEmployee
from dbo.DimEmployee as de
group by rollup (de.DepartmentName, de.Gender)
order by grouping(de.DepartmentName), de.DepartmentName, grouping(de.Gender), de.Gender;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>Department</th><th>Gender</th><th>CountOfEmployee</th></tr></thead><tbody><tr><td>Engineering</td><td>F</td><td style="text-align:right">1</td></tr><tr><td>Engineering</td><td>M</td><td style="text-align:right">3</td></tr><tr><td>Engineering</td><td>All genders</td><td style="text-align:right">4</td></tr><tr><td>Marketing</td><td>F</td><td style="text-align:right">2</td></tr><tr><td>Marketing</td><td>M</td><td style="text-align:right">1</td></tr><tr><td>Marketing</td><td>All genders</td><td style="text-align:right">3</td></tr><tr><td>Sales</td><td>F</td><td style="text-align:right">2</td></tr><tr><td>Sales</td><td>M</td><td style="text-align:right">3</td></tr><tr><td>Sales</td><td>All genders</td><td style="text-align:right">5</td></tr><tr><td>All departments</td><td>All genders</td><td style="text-align:right">12</td></tr></tbody></table>

> The lecture uses the older `WITH CUBE` / `WITH ROLLUP` syntax. The newer `ROLLUP(a, b)` and `CUBE(a, b)` forms (used above in 6.4) mean the same thing and are the recommended style in current SQL Server versions.

<a id="s7"></a>
## 7. Practice: Northwind and pubs

### Exercise 7.1: customers per country, most to fewest

*Write a query that shows the number of customers per country, from the largest count to the smallest.*

In [ ]:
use Northwind
go
select
    c.Country,
    count(1) as CountOfCustomers
from dbo.Customers as c
group by c.Country
order by CountOfCustomers desc;     -- desc = descending (largest first)

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>Country</th><th>CountOfCustomers</th></tr></thead><tbody><tr><td>France</td><td style="text-align:right">4</td></tr><tr><td>UK</td><td style="text-align:right">4</td></tr><tr><td>Germany</td><td style="text-align:right">3</td></tr><tr><td>Mexico</td><td style="text-align:right">3</td></tr><tr><td>Brazil</td><td style="text-align:right">2</td></tr><tr><td>Spain</td><td style="text-align:right">2</td></tr><tr><td>Sweden</td><td style="text-align:right">2</td></tr><tr><td>Argentina</td><td style="text-align:right">1</td></tr><tr><td>Austria</td><td style="text-align:right">1</td></tr><tr><td>Canada</td><td style="text-align:right">1</td></tr><tr><td>Switzerland</td><td style="text-align:right">1</td></tr></tbody></table>

### Exercise 7.2: customers per country and city, with all subtotals

*Show the number of customers per country and city, including the country subtotals and the grand total.*

`ROLLUP(Country, City)` is the right tool because the data is hierarchical (a city belongs to a country).

In [ ]:
select
    c.Country,
    c.City,
    count(1) as CountOfCustomers
from dbo.Customers as c
group by c.Country, c.City with rollup
order by c.Country, c.City;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>Country</th><th>City</th><th>CountOfCustomers</th></tr></thead><tbody><tr><td><i>NULL</i></td><td><i>NULL</i></td><td style="text-align:right">24</td></tr><tr><td>Argentina</td><td><i>NULL</i></td><td style="text-align:right">1</td></tr><tr><td>Argentina</td><td>Buenos Aires</td><td style="text-align:right">1</td></tr><tr><td>Austria</td><td><i>NULL</i></td><td style="text-align:right">1</td></tr><tr><td>Austria</td><td>Graz</td><td style="text-align:right">1</td></tr><tr><td>Brazil</td><td><i>NULL</i></td><td style="text-align:right">2</td></tr><tr><td>Brazil</td><td>São Paulo</td><td style="text-align:right">2</td></tr><tr><td>Canada</td><td><i>NULL</i></td><td style="text-align:right">1</td></tr><tr><td>Canada</td><td>Tsawwassen</td><td style="text-align:right">1</td></tr><tr><td>France</td><td><i>NULL</i></td><td style="text-align:right">4</td></tr><tr><td>France</td><td>Lille</td><td style="text-align:right">1</td></tr><tr><td>France</td><td>Marseille</td><td style="text-align:right">1</td></tr><tr><td>France</td><td>Nantes</td><td style="text-align:right">1</td></tr><tr><td>France</td><td>Strasbourg</td><td style="text-align:right">1</td></tr><tr><td>Germany</td><td><i>NULL</i></td><td style="text-align:right">3</td></tr><tr><td>Germany</td><td>Aachen</td><td style="text-align:right">1</td></tr><tr><td>Germany</td><td>Berlin</td><td style="text-align:right">1</td></tr><tr><td>Germany</td><td>Mannheim</td><td style="text-align:right">1</td></tr><tr><td>Mexico</td><td><i>NULL</i></td><td style="text-align:right">3</td></tr><tr><td>Mexico</td><td>México D.F.</td><td style="text-align:right">3</td></tr><tr><td>Spain</td><td><i>NULL</i></td><td style="text-align:right">2</td></tr><tr><td>Spain</td><td>Madrid</td><td style="text-align:right">2</td></tr><tr><td>Sweden</td><td><i>NULL</i></td><td style="text-align:right">2</td></tr><tr><td>Sweden</td><td>Bräcke</td><td style="text-align:right">1</td></tr><tr><td>Sweden</td><td>Luleå</td><td style="text-align:right">1</td></tr><tr><td>Switzerland</td><td><i>NULL</i></td><td style="text-align:right">1</td></tr><tr><td>Switzerland</td><td>Bern</td><td style="text-align:right">1</td></tr><tr><td>UK</td><td><i>NULL</i></td><td style="text-align:right">4</td></tr><tr><td>UK</td><td>London</td><td style="text-align:right">4</td></tr></tbody></table>

### Exercise 7.3: only the country rows and the grand total

*Show customers per country and city with the Grand Total and the country subtotals only.*

Here `ROLLUP` would give extra rows we do not need, so we spell out the groupings with `GROUPING SETS`.

In [ ]:
select
    c.Country,
    c.City,
    count(1) as CountOfCustomers
from dbo.Customers as c
group by grouping sets (
        (c.Country, c.City),
        (),                      -- grand total
        (c.Country)              -- country subtotal
)
order by c.Country, c.City;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>Country</th><th>City</th><th>CountOfCustomers</th></tr></thead><tbody><tr><td><i>NULL</i></td><td><i>NULL</i></td><td style="text-align:right">24</td></tr><tr><td>Argentina</td><td><i>NULL</i></td><td style="text-align:right">1</td></tr><tr><td>Argentina</td><td>Buenos Aires</td><td style="text-align:right">1</td></tr><tr><td>Austria</td><td><i>NULL</i></td><td style="text-align:right">1</td></tr><tr><td>Austria</td><td>Graz</td><td style="text-align:right">1</td></tr><tr><td>Brazil</td><td><i>NULL</i></td><td style="text-align:right">2</td></tr><tr><td>Brazil</td><td>São Paulo</td><td style="text-align:right">2</td></tr><tr><td>Canada</td><td><i>NULL</i></td><td style="text-align:right">1</td></tr><tr><td>Canada</td><td>Tsawwassen</td><td style="text-align:right">1</td></tr><tr><td>France</td><td><i>NULL</i></td><td style="text-align:right">4</td></tr><tr><td>France</td><td>Lille</td><td style="text-align:right">1</td></tr><tr><td>France</td><td>Marseille</td><td style="text-align:right">1</td></tr><tr><td>France</td><td>Nantes</td><td style="text-align:right">1</td></tr><tr><td>France</td><td>Strasbourg</td><td style="text-align:right">1</td></tr><tr><td>Germany</td><td><i>NULL</i></td><td style="text-align:right">3</td></tr><tr><td>Germany</td><td>Aachen</td><td style="text-align:right">1</td></tr><tr><td>Germany</td><td>Berlin</td><td style="text-align:right">1</td></tr><tr><td>Germany</td><td>Mannheim</td><td style="text-align:right">1</td></tr><tr><td>Mexico</td><td><i>NULL</i></td><td style="text-align:right">3</td></tr><tr><td>Mexico</td><td>México D.F.</td><td style="text-align:right">3</td></tr><tr><td>Spain</td><td><i>NULL</i></td><td style="text-align:right">2</td></tr><tr><td>Spain</td><td>Madrid</td><td style="text-align:right">2</td></tr><tr><td>Sweden</td><td><i>NULL</i></td><td style="text-align:right">2</td></tr><tr><td>Sweden</td><td>Bräcke</td><td style="text-align:right">1</td></tr><tr><td>Sweden</td><td>Luleå</td><td style="text-align:right">1</td></tr><tr><td>Switzerland</td><td><i>NULL</i></td><td style="text-align:right">1</td></tr><tr><td>Switzerland</td><td>Bern</td><td style="text-align:right">1</td></tr><tr><td>UK</td><td><i>NULL</i></td><td style="text-align:right">4</td></tr><tr><td>UK</td><td>London</td><td style="text-align:right">4</td></tr></tbody></table>

### Exercise 7.4: authors with and without a contract

*Count the authors we have a contract with and those we do not.*

**Version 1: `GROUP BY` (result in rows).** `contract` is a bit column: 1 = contract, 0 = no contract.

In [ ]:
use pubs
go
select * from pubs.dbo.authors as a;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>au_id</th><th>au_lname</th><th>au_fname</th><th>state</th><th>contract</th></tr></thead><tbody><tr><td>172-32-1176</td><td>White</td><td>Johnson</td><td>CA</td><td style="text-align:right">1</td></tr><tr><td>213-46-8915</td><td>Green</td><td>Marjorie</td><td>CA</td><td style="text-align:right">1</td></tr><tr><td>238-95-7766</td><td>Carson</td><td>Cheryl</td><td>CA</td><td style="text-align:right">1</td></tr><tr><td>267-41-2394</td><td>O&#x27;Leary</td><td>Michael</td><td>CA</td><td style="text-align:right">1</td></tr><tr><td>274-80-9391</td><td>Straight</td><td>Dean</td><td>CA</td><td style="text-align:right">1</td></tr><tr><td>341-22-1782</td><td>Smith</td><td>Meander</td><td>KS</td><td style="text-align:right">0</td></tr><tr><td>409-56-7008</td><td>Bennet</td><td>Abraham</td><td>CA</td><td style="text-align:right">1</td></tr><tr><td>427-17-2319</td><td>Dull</td><td>Ann</td><td>CA</td><td style="text-align:right">1</td></tr><tr><td>472-27-2349</td><td>Gringlesby</td><td>Burt</td><td>CA</td><td style="text-align:right">1</td></tr><tr><td>486-29-1786</td><td>Locksley</td><td>Charlene</td><td>CA</td><td style="text-align:right">1</td></tr><tr><td>527-72-3246</td><td>Greene</td><td>Morningstar</td><td>TN</td><td style="text-align:right">0</td></tr><tr><td>648-92-1872</td><td>Blotchet-Halls</td><td>Reginald</td><td>OR</td><td style="text-align:right">1</td></tr><tr><td>672-71-3249</td><td>Yokomoto</td><td>Akiko</td><td>CA</td><td style="text-align:right">1</td></tr><tr><td>712-45-1867</td><td>del Castillo</td><td>Innes</td><td>MI</td><td style="text-align:right">1</td></tr><tr><td>722-51-5454</td><td>DeFrance</td><td>Michel</td><td>IN</td><td style="text-align:right">1</td></tr><tr><td>724-08-9931</td><td>Stringer</td><td>Dirk</td><td>CA</td><td style="text-align:right">0</td></tr><tr><td>724-80-9391</td><td>MacFeather</td><td>Stearns</td><td>CA</td><td style="text-align:right">1</td></tr><tr><td>756-30-7391</td><td>Karsen</td><td>Livia</td><td>CA</td><td style="text-align:right">1</td></tr><tr><td>807-91-6654</td><td>Panteley</td><td>Sylvia</td><td>MD</td><td style="text-align:right">1</td></tr><tr><td>846-92-7186</td><td>Hunter</td><td>Sheryl</td><td>CA</td><td style="text-align:right">1</td></tr><tr><td>893-72-1158</td><td>McBadden</td><td>Heather</td><td>CA</td><td style="text-align:right">0</td></tr><tr><td>899-46-2035</td><td>Ringer</td><td>Anne</td><td>UT</td><td style="text-align:right">1</td></tr><tr><td>998-72-3567</td><td>Ringer</td><td>Albert</td><td>UT</td><td style="text-align:right">1</td></tr></tbody></table>

In [ ]:
select
    a.[contract],
    count(1) as CountOfAuthors
from pubs..authors as a
group by a.[contract];

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>contract</th><th>CountOfAuthors</th></tr></thead><tbody><tr><td style="text-align:right">1</td><td style="text-align:right">19</td></tr><tr><td style="text-align:right">0</td><td style="text-align:right">4</td></tr></tbody></table>

**Version 2: conditional counting with `CASE` (result in columns).**

The requested layout is one row with two columns:

| WithContract | WithoutContract |
|---|---|

`COUNT` skips `NULL`s, and a `CASE` without `ELSE` returns `NULL` when the condition is false. So `COUNT(CASE WHEN condition THEN 1 END)` counts only the rows that meet the condition. This technique is called **conditional aggregation** and is the standard way to turn categories into columns.

In [ ]:
select
    count(case when a.[contract] = 1 then 1 end) as WithContract,
    count(case when a.[contract] = 0 then 1 end) as WithoutContract
from pubs..authors as a;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>WithContract</th><th>WithoutContract</th></tr></thead><tbody><tr><td style="text-align:right">19</td><td style="text-align:right">4</td></tr></tbody></table>

<a id="s8"></a>
## 8. Joins

### Combining tables: two directions

| Direction | Operators | Result |
|---|---|---|
| **Horizontal** (add columns) | `JOIN`: `INNER`, `OUTER` (`LEFT`/`RIGHT`/`FULL`), `CROSS` | wider rows built from matching records |
| **Vertical** (add rows) | `UNION`, `UNION ALL`, `EXCEPT`, `INTERSECT` | longer result stacked from several queries |

This session covers the horizontal case, starting with `INNER JOIN`. Outer joins, cross joins and the set operators come next.

### Why joins exist
A well-designed database splits data across tables to avoid repetition: `titles` stores only a publisher *id*, and the publisher's name lives once in `publishers`. A join reassembles the information by matching the key columns.

### `INNER JOIN`
Returns only the rows that have a **match in both tables** under the `ON` condition. Rows without a partner are dropped.

```
FROM   tableA AS a
INNER JOIN tableB AS b
ON     a.key = b.key
```

* Always use **table aliases** (`t`, `p`) and prefix every column with its alias. It makes the query shorter and avoids "ambiguous column name" errors.
* `INNER` is optional: `JOIN` alone means `INNER JOIN`.

The two tables we join first:

In [ ]:
use pubs
go
select * from dbo.publishers as p
select * from dbo.titles as t

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>pub_id</th><th>pub_name</th><th>city</th><th>state</th><th>country</th></tr></thead><tbody><tr><td>0736</td><td>New Moon Books</td><td>Boston</td><td>MA</td><td>USA</td></tr><tr><td>0877</td><td>Binnet &amp; Hardley</td><td>Washington</td><td>DC</td><td>USA</td></tr><tr><td>1389</td><td>Algodata Infosystems</td><td>Berkeley</td><td>CA</td><td>USA</td></tr><tr><td>1622</td><td>Five Lakes Publishing</td><td>Chicago</td><td>IL</td><td>USA</td></tr><tr><td>1756</td><td>Ramona Publishers</td><td>Dallas</td><td>TX</td><td>USA</td></tr><tr><td>9901</td><td>GGG&amp;G</td><td>Munchen</td><td><i>NULL</i></td><td>Germany</td></tr><tr><td>9952</td><td>Scootney Books</td><td>New York</td><td>NY</td><td>USA</td></tr><tr><td>9999</td><td>Lucerne Publishing</td><td>Paris</td><td><i>NULL</i></td><td>France</td></tr></tbody></table>

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>title_id</th><th>title</th><th>type</th><th>pub_id</th><th>price</th><th>pubdate</th></tr></thead><tbody><tr><td>BU1032</td><td>The Busy Executive&#x27;s Database Guide</td><td>business</td><td>1389</td><td style="text-align:right">19.99</td><td>1991-06-12</td></tr><tr><td>BU1111</td><td>Cooking with Computers: Surreptitious Balance Sheets</td><td>business</td><td>1389</td><td style="text-align:right">11.95</td><td>1991-06-09</td></tr><tr><td>BU2075</td><td>You Can Combat Computer Stress!</td><td>business</td><td>0736</td><td style="text-align:right">2.99</td><td>1991-06-30</td></tr><tr><td>BU7832</td><td>Straight Talk About Computers</td><td>business</td><td>1389</td><td style="text-align:right">19.99</td><td>1991-06-22</td></tr><tr><td>MC2222</td><td>Silicon Valley Gastronomic Treats</td><td>mod_cook</td><td>0877</td><td style="text-align:right">19.99</td><td>1991-06-09</td></tr><tr><td>MC3021</td><td>The Gourmet Microwave</td><td>mod_cook</td><td>0877</td><td style="text-align:right">2.99</td><td>1991-06-18</td></tr><tr><td>MC3026</td><td>The Psychology of Computer Cooking</td><td>UNDECIDED</td><td>0877</td><td style="text-align:right"><i>NULL</i></td><td>1991-06-12</td></tr><tr><td>PC1035</td><td>But Is It User Friendly?</td><td>popular_comp</td><td>1389</td><td style="text-align:right">22.95</td><td>1991-06-30</td></tr><tr><td>PC8888</td><td>Secrets of Silicon Valley</td><td>popular_comp</td><td>1389</td><td style="text-align:right">20.00</td><td>1994-06-12</td></tr><tr><td>PC9999</td><td>Net Etiquette</td><td>popular_comp</td><td>1389</td><td style="text-align:right"><i>NULL</i></td><td>1994-06-12</td></tr><tr><td>PS1372</td><td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td><td>psychology</td><td>0877</td><td style="text-align:right">21.59</td><td>1991-10-21</td></tr><tr><td>PS2091</td><td>Is Anger the Enemy?</td><td>psychology</td><td>0736</td><td style="text-align:right">10.95</td><td>1991-06-15</td></tr><tr><td>PS2106</td><td>Life Without Fear</td><td>psychology</td><td>0736</td><td style="text-align:right">7.00</td><td>1991-10-05</td></tr><tr><td>PS3333</td><td>Prolonged Data Deprivation: Four Case Studies</td><td>psychology</td><td>0736</td><td style="text-align:right">19.99</td><td>1991-06-12</td></tr><tr><td>PS7777</td><td>Emotional Security: A New Algorithm</td><td>psychology</td><td>0736</td><td style="text-align:right">7.99</td><td>1991-06-12</td></tr><tr><td>TC3218</td><td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td><td>trad_cook</td><td>0877</td><td style="text-align:right">20.95</td><td>1991-10-21</td></tr><tr><td>TC4203</td><td>Fifty Years in Buckingham Palace Kitchens</td><td>trad_cook</td><td>0877</td><td style="text-align:right">11.95</td><td>1991-06-12</td></tr><tr><td>TC7777</td><td>Sushi, Anyone?</td><td>trad_cook</td><td>0877</td><td style="text-align:right">14.99</td><td>1991-06-12</td></tr></tbody></table>

### 8.1 Basic `INNER JOIN`

Show each title with its publisher's name. `pub_id` is the key that links the tables (primary key in `publishers`, foreign key in `titles`).

In [ ]:
select
    t.title_id,     -- titles
    t.title,        -- titles
    t.price,        -- titles
    t.[type],       -- titles
    p.pub_name      -- publishers
from dbo.publishers as p
inner join dbo.titles as t
on p.pub_id = t.pub_id;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>title_id</th><th>title</th><th>price</th><th>type</th><th>pub_name</th></tr></thead><tbody><tr><td>BU1032</td><td>The Busy Executive&#x27;s Database Guide</td><td style="text-align:right">19.99</td><td>business</td><td>Algodata Infosystems</td></tr><tr><td>BU1111</td><td>Cooking with Computers: Surreptitious Balance Sheets</td><td style="text-align:right">11.95</td><td>business</td><td>Algodata Infosystems</td></tr><tr><td>BU2075</td><td>You Can Combat Computer Stress!</td><td style="text-align:right">2.99</td><td>business</td><td>New Moon Books</td></tr><tr><td>BU7832</td><td>Straight Talk About Computers</td><td style="text-align:right">19.99</td><td>business</td><td>Algodata Infosystems</td></tr><tr><td>MC2222</td><td>Silicon Valley Gastronomic Treats</td><td style="text-align:right">19.99</td><td>mod_cook</td><td>Binnet &amp; Hardley</td></tr><tr><td>MC3021</td><td>The Gourmet Microwave</td><td style="text-align:right">2.99</td><td>mod_cook</td><td>Binnet &amp; Hardley</td></tr><tr><td>MC3026</td><td>The Psychology of Computer Cooking</td><td style="text-align:right"><i>NULL</i></td><td>UNDECIDED</td><td>Binnet &amp; Hardley</td></tr><tr><td>PC1035</td><td>But Is It User Friendly?</td><td style="text-align:right">22.95</td><td>popular_comp</td><td>Algodata Infosystems</td></tr><tr><td>PC8888</td><td>Secrets of Silicon Valley</td><td style="text-align:right">20.00</td><td>popular_comp</td><td>Algodata Infosystems</td></tr><tr><td>PC9999</td><td>Net Etiquette</td><td style="text-align:right"><i>NULL</i></td><td>popular_comp</td><td>Algodata Infosystems</td></tr><tr><td>PS1372</td><td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td><td style="text-align:right">21.59</td><td>psychology</td><td>Binnet &amp; Hardley</td></tr><tr><td>PS2091</td><td>Is Anger the Enemy?</td><td style="text-align:right">10.95</td><td>psychology</td><td>New Moon Books</td></tr><tr><td>PS2106</td><td>Life Without Fear</td><td style="text-align:right">7.00</td><td>psychology</td><td>New Moon Books</td></tr><tr><td>PS3333</td><td>Prolonged Data Deprivation: Four Case Studies</td><td style="text-align:right">19.99</td><td>psychology</td><td>New Moon Books</td></tr><tr><td>PS7777</td><td>Emotional Security: A New Algorithm</td><td style="text-align:right">7.99</td><td>psychology</td><td>New Moon Books</td></tr><tr><td>TC3218</td><td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td><td style="text-align:right">20.95</td><td>trad_cook</td><td>Binnet &amp; Hardley</td></tr><tr><td>TC4203</td><td>Fifty Years in Buckingham Palace Kitchens</td><td style="text-align:right">11.95</td><td>trad_cook</td><td>Binnet &amp; Hardley</td></tr><tr><td>TC7777</td><td>Sushi, Anyone?</td><td style="text-align:right">14.99</td><td>trad_cook</td><td>Binnet &amp; Hardley</td></tr></tbody></table>

### 8.2 Join plus filter

A useful way to plan a join query: **A** = the columns you want to *show* (select list), **B** = the columns you want to *filter on*. A and B may come from different tables, so you join to reach both.

Here: show title details (A, from `titles`) for publishers located in California (B, from `publishers`).

In [ ]:
-- Select list => A  (titles)
-- Filter      => B  (publishers.state)
select
    t.title_id,
    t.title,
    t.price,
    t.[type]
from dbo.titles as t
inner join dbo.publishers as p
on t.pub_id = p.pub_id
where p.[state] = 'CA';

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>title_id</th><th>title</th><th>price</th><th>type</th></tr></thead><tbody><tr><td>BU1032</td><td>The Busy Executive&#x27;s Database Guide</td><td style="text-align:right">19.99</td><td>business</td></tr><tr><td>BU1111</td><td>Cooking with Computers: Surreptitious Balance Sheets</td><td style="text-align:right">11.95</td><td>business</td></tr><tr><td>BU7832</td><td>Straight Talk About Computers</td><td style="text-align:right">19.99</td><td>business</td></tr><tr><td>PC1035</td><td>But Is It User Friendly?</td><td style="text-align:right">22.95</td><td>popular_comp</td></tr><tr><td>PC8888</td><td>Secrets of Silicon Valley</td><td style="text-align:right">20.00</td><td>popular_comp</td></tr><tr><td>PC9999</td><td>Net Etiquette</td><td style="text-align:right"><i>NULL</i></td><td>popular_comp</td></tr></tbody></table>

### 8.3 `JOIN` without the word `INNER`

Identical meaning and identical result. (Output matches 8.1.)

In [ ]:
-- A, B
select
    t.title_id,
    t.title,
    t.price,
    t.[type],
    p.pub_name
from dbo.publishers as p
join dbo.titles as t
on p.pub_id = t.pub_id;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>title_id</th><th>title</th><th>price</th><th>type</th><th>pub_name</th></tr></thead><tbody><tr><td>BU1032</td><td>The Busy Executive&#x27;s Database Guide</td><td style="text-align:right">19.99</td><td>business</td><td>Algodata Infosystems</td></tr><tr><td>BU1111</td><td>Cooking with Computers: Surreptitious Balance Sheets</td><td style="text-align:right">11.95</td><td>business</td><td>Algodata Infosystems</td></tr><tr><td>BU2075</td><td>You Can Combat Computer Stress!</td><td style="text-align:right">2.99</td><td>business</td><td>New Moon Books</td></tr><tr><td>BU7832</td><td>Straight Talk About Computers</td><td style="text-align:right">19.99</td><td>business</td><td>Algodata Infosystems</td></tr><tr><td>MC2222</td><td>Silicon Valley Gastronomic Treats</td><td style="text-align:right">19.99</td><td>mod_cook</td><td>Binnet &amp; Hardley</td></tr><tr><td>MC3021</td><td>The Gourmet Microwave</td><td style="text-align:right">2.99</td><td>mod_cook</td><td>Binnet &amp; Hardley</td></tr><tr><td>MC3026</td><td>The Psychology of Computer Cooking</td><td style="text-align:right"><i>NULL</i></td><td>UNDECIDED</td><td>Binnet &amp; Hardley</td></tr><tr><td>PC1035</td><td>But Is It User Friendly?</td><td style="text-align:right">22.95</td><td>popular_comp</td><td>Algodata Infosystems</td></tr><tr><td>PC8888</td><td>Secrets of Silicon Valley</td><td style="text-align:right">20.00</td><td>popular_comp</td><td>Algodata Infosystems</td></tr><tr><td>PC9999</td><td>Net Etiquette</td><td style="text-align:right"><i>NULL</i></td><td>popular_comp</td><td>Algodata Infosystems</td></tr><tr><td>PS1372</td><td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td><td style="text-align:right">21.59</td><td>psychology</td><td>Binnet &amp; Hardley</td></tr><tr><td>PS2091</td><td>Is Anger the Enemy?</td><td style="text-align:right">10.95</td><td>psychology</td><td>New Moon Books</td></tr><tr><td>PS2106</td><td>Life Without Fear</td><td style="text-align:right">7.00</td><td>psychology</td><td>New Moon Books</td></tr><tr><td>PS3333</td><td>Prolonged Data Deprivation: Four Case Studies</td><td style="text-align:right">19.99</td><td>psychology</td><td>New Moon Books</td></tr><tr><td>PS7777</td><td>Emotional Security: A New Algorithm</td><td style="text-align:right">7.99</td><td>psychology</td><td>New Moon Books</td></tr><tr><td>TC3218</td><td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td><td style="text-align:right">20.95</td><td>trad_cook</td><td>Binnet &amp; Hardley</td></tr><tr><td>TC4203</td><td>Fifty Years in Buckingham Palace Kitchens</td><td style="text-align:right">11.95</td><td>trad_cook</td><td>Binnet &amp; Hardley</td></tr><tr><td>TC7777</td><td>Sushi, Anyone?</td><td style="text-align:right">14.99</td><td>trad_cook</td><td>Binnet &amp; Hardley</td></tr></tbody></table>

### 8.4 Joining three tables: many-to-many

One book can have several authors and one author can write several books. That relationship cannot live in either table, so a **bridge table** (`titleauthor`) stores the pairs. To reach the author names you chain two joins: `titles → titleauthor → authors`.

`CONCAT_WS(' ', first, last)` builds the full name: the first argument is the separator, the rest are the pieces. Unlike the `+` operator, it **skips `NULL`** arguments instead of turning the whole name into `NULL`.

In [ ]:
select
    t.title_id,
    t.title,
    t.price,
    CONCAT_WS(' ', a.au_fname, a.au_lname) as FullName
from dbo.titles as t
inner join dbo.titleauthor as ta
on t.title_id = ta.title_id
inner join dbo.authors as a
on ta.au_id = a.au_id;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>title_id</th><th>title</th><th>price</th><th>FullName</th></tr></thead><tbody><tr><td>PS3333</td><td>Prolonged Data Deprivation: Four Case Studies</td><td style="text-align:right">19.99</td><td>Johnson White</td></tr><tr><td>BU1032</td><td>The Busy Executive&#x27;s Database Guide</td><td style="text-align:right">19.99</td><td>Marjorie Green</td></tr><tr><td>BU2075</td><td>You Can Combat Computer Stress!</td><td style="text-align:right">2.99</td><td>Marjorie Green</td></tr><tr><td>PC1035</td><td>But Is It User Friendly?</td><td style="text-align:right">22.95</td><td>Cheryl Carson</td></tr><tr><td>BU1111</td><td>Cooking with Computers: Surreptitious Balance Sheets</td><td style="text-align:right">11.95</td><td>Michael O&#x27;Leary</td></tr><tr><td>TC7777</td><td>Sushi, Anyone?</td><td style="text-align:right">14.99</td><td>Michael O&#x27;Leary</td></tr><tr><td>BU7832</td><td>Straight Talk About Computers</td><td style="text-align:right">19.99</td><td>Dean Straight</td></tr><tr><td>BU1032</td><td>The Busy Executive&#x27;s Database Guide</td><td style="text-align:right">19.99</td><td>Abraham Bennet</td></tr><tr><td>PC8888</td><td>Secrets of Silicon Valley</td><td style="text-align:right">20.00</td><td>Ann Dull</td></tr><tr><td>TC7777</td><td>Sushi, Anyone?</td><td style="text-align:right">14.99</td><td>Burt Gringlesby</td></tr><tr><td>PC9999</td><td>Net Etiquette</td><td style="text-align:right"><i>NULL</i></td><td>Charlene Locksley</td></tr><tr><td>PS7777</td><td>Emotional Security: A New Algorithm</td><td style="text-align:right">7.99</td><td>Charlene Locksley</td></tr><tr><td>TC4203</td><td>Fifty Years in Buckingham Palace Kitchens</td><td style="text-align:right">11.95</td><td>Reginald Blotchet-Halls</td></tr><tr><td>TC7777</td><td>Sushi, Anyone?</td><td style="text-align:right">14.99</td><td>Akiko Yokomoto</td></tr><tr><td>MC2222</td><td>Silicon Valley Gastronomic Treats</td><td style="text-align:right">19.99</td><td>Innes del Castillo</td></tr><tr><td>MC3021</td><td>The Gourmet Microwave</td><td style="text-align:right">2.99</td><td>Michel DeFrance</td></tr><tr><td>BU1111</td><td>Cooking with Computers: Surreptitious Balance Sheets</td><td style="text-align:right">11.95</td><td>Stearns MacFeather</td></tr><tr><td>PS1372</td><td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td><td style="text-align:right">21.59</td><td>Stearns MacFeather</td></tr><tr><td>PS1372</td><td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td><td style="text-align:right">21.59</td><td>Livia Karsen</td></tr><tr><td>TC3218</td><td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td><td style="text-align:right">20.95</td><td>Sylvia Panteley</td></tr><tr><td>PC8888</td><td>Secrets of Silicon Valley</td><td style="text-align:right">20.00</td><td>Sheryl Hunter</td></tr><tr><td>MC3021</td><td>The Gourmet Microwave</td><td style="text-align:right">2.99</td><td>Anne Ringer</td></tr><tr><td>PS2091</td><td>Is Anger the Enemy?</td><td style="text-align:right">10.95</td><td>Anne Ringer</td></tr><tr><td>PS2091</td><td>Is Anger the Enemy?</td><td style="text-align:right">10.95</td><td>Albert Ringer</td></tr><tr><td>PS2106</td><td>Life Without Fear</td><td style="text-align:right">7.00</td><td>Albert Ringer</td></tr></tbody></table>

Only titles that have at least one matching row in `titleauthor` appear. `MC3026` has no author row, so the inner join drops it: 17 of the 18 titles are matched, and the 25 result rows come from titles with several authors appearing once per author (a title with two authors appears twice).

### 8.5 Join + `GROUP BY` + `HAVING`

Which titles have **more than one author**? Join, group per title, count authors, keep the groups with count above 1.

In [ ]:
select
    t.title_id,
    t.title,
    count(ta.au_id) as CountOfAuthors
from dbo.titles as t
inner join dbo.titleauthor as ta
on t.title_id = ta.title_id
group by
    t.title_id,
    t.title
having count(ta.au_id) > 1
order by CountOfAuthors desc;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>title_id</th><th>title</th><th>CountOfAuthors</th></tr></thead><tbody><tr><td>TC7777</td><td>Sushi, Anyone?</td><td style="text-align:right">3</td></tr><tr><td>BU1032</td><td>The Busy Executive&#x27;s Database Guide</td><td style="text-align:right">2</td></tr><tr><td>BU1111</td><td>Cooking with Computers: Surreptitious Balance Sheets</td><td style="text-align:right">2</td></tr><tr><td>MC3021</td><td>The Gourmet Microwave</td><td style="text-align:right">2</td></tr><tr><td>PC8888</td><td>Secrets of Silicon Valley</td><td style="text-align:right">2</td></tr><tr><td>PS1372</td><td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td><td style="text-align:right">2</td></tr><tr><td>PS2091</td><td>Is Anger the Enemy?</td><td style="text-align:right">2</td></tr></tbody></table>

The logical order from section 3 now includes the join: `FROM` → `JOIN` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `ORDER BY`.

The `sales` table, used in the next examples:

In [ ]:
select * from dbo.sales as s

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>stor_id</th><th>ord_num</th><th>ord_date</th><th>qty</th><th>title_id</th></tr></thead><tbody><tr><td>6380</td><td>6871</td><td>1994-09-14</td><td style="text-align:right">5</td><td>BU1032</td></tr><tr><td>6380</td><td>722a</td><td>1994-09-13</td><td style="text-align:right">3</td><td>PS2091</td></tr><tr><td>7066</td><td>A2976</td><td>1993-05-24</td><td style="text-align:right">50</td><td>PC8888</td></tr><tr><td>7066</td><td>QQ2299</td><td>1993-10-28</td><td style="text-align:right">15</td><td>BU7832</td></tr><tr><td>7067</td><td>D4482</td><td>1994-09-14</td><td style="text-align:right">10</td><td>PS2091</td></tr><tr><td>7067</td><td>P2121</td><td>1992-06-15</td><td style="text-align:right">40</td><td>TC3218</td></tr><tr><td>7067</td><td>P2121</td><td>1992-06-15</td><td style="text-align:right">20</td><td>TC4203</td></tr><tr><td>7067</td><td>P2121</td><td>1992-06-15</td><td style="text-align:right">20</td><td>TC7777</td></tr><tr><td>7131</td><td>N914008</td><td>1994-09-14</td><td style="text-align:right">20</td><td>PS2091</td></tr><tr><td>7131</td><td>N914014</td><td>1994-09-14</td><td style="text-align:right">25</td><td>MC3021</td></tr><tr><td>7131</td><td>P3087a</td><td>1993-05-29</td><td style="text-align:right">20</td><td>PS1372</td></tr><tr><td>7131</td><td>P3087a</td><td>1993-05-29</td><td style="text-align:right">25</td><td>PS2106</td></tr><tr><td>7131</td><td>P3087a</td><td>1993-05-29</td><td style="text-align:right">15</td><td>PS3333</td></tr><tr><td>7131</td><td>P3087a</td><td>1993-05-29</td><td style="text-align:right">25</td><td>PS7777</td></tr><tr><td>7896</td><td>QA7442.3</td><td>1994-09-13</td><td style="text-align:right">75</td><td>PS2091</td></tr><tr><td>7896</td><td>TQ456</td><td>1993-12-12</td><td style="text-align:right">10</td><td>MC2222</td></tr><tr><td>7896</td><td>X999</td><td>1993-02-21</td><td style="text-align:right">35</td><td>BU2075</td></tr><tr><td>8042</td><td>423LL922</td><td>1994-09-14</td><td style="text-align:right">15</td><td>MC3021</td></tr><tr><td>8042</td><td>423LL930</td><td>1994-09-14</td><td style="text-align:right">10</td><td>BU1032</td></tr><tr><td>8042</td><td>P723</td><td>1993-03-11</td><td style="text-align:right">25</td><td>BU1111</td></tr><tr><td>8042</td><td>QA879.1</td><td>1993-05-22</td><td style="text-align:right">30</td><td>PC1035</td></tr></tbody></table>

<a id="s9"></a>
## 9. `TOP ... WITH TIES` and `FORMAT`

### `TOP n`
Returns only the first *n* rows of the **sorted** result. It needs `ORDER BY`, otherwise "first" has no meaning.

### `TOP n WITH TIES`
If the row at position *n* has the same sort value as the rows after it, `WITH TIES` keeps those too, so you never cut a tie in half unfairly. The result can contain more than *n* rows. `WITH TIES` **requires** `ORDER BY`.

### Example: the top 3 best-selling titles by quantity

In the data, positions 3 and 4 both sold 40 copies, so `WITH TIES` returns four rows.

In [ ]:
select top 3 with ties
    t.title_id,
    t.title,
    sum(s.qty) as TotalOrders
from dbo.titles as t
inner join dbo.sales as s
on t.title_id = s.title_id
group by
    t.title_id,
    t.title
order by TotalOrders desc;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>title_id</th><th>title</th><th>TotalOrders</th></tr></thead><tbody><tr><td>PS2091</td><td>Is Anger the Enemy?</td><td style="text-align:right">108</td></tr><tr><td>PC8888</td><td>Secrets of Silicon Valley</td><td style="text-align:right">50</td></tr><tr><td>MC3021</td><td>The Gourmet Microwave</td><td style="text-align:right">40</td></tr><tr><td>TC3218</td><td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td><td style="text-align:right">40</td></tr></tbody></table>

### `FORMAT(value, pattern)`

Turns a number (or date) into **text** using a .NET format pattern. In `'#,#.#'`, `#` is an optional digit, `,` adds thousands separators and `.#` keeps at most one decimal.

Important detail: `FORMAT` returns **text**, so sorting on the formatted column would sort alphabetically ("9" after "10"). That is why the query below sorts by the original numeric expression `sum(s.qty * t.price)`, not by the alias.

This query is a three-table join with a calculated column: total sales value (quantity × price) per publisher.

In [ ]:
select
    p.pub_name,
    format(sum(s.qty * t.price), '#,#.#') as TotalSales
from dbo.publishers as p
inner join dbo.titles as t
on p.pub_id = t.pub_id
inner join dbo.sales as s
on t.title_id = s.title_id
group by p.pub_name
order by sum(s.qty * t.price) desc;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>pub_name</th><th>TotalSales</th></tr></thead><tbody><tr><td>Algodata Infosystems</td><td style="text-align:right">2,587</td></tr><tr><td>Binnet &amp; Hardley</td><td style="text-align:right">2,128.1</td></tr><tr><td>New Moon Books</td><td style="text-align:right">1,961.9</td></tr></tbody></table>

### A four-table join (generated by the query designer)

SSMS's visual query designer produces this style: no aliases, full table names. It works the same way, but aliases (as above) are easier to read. Chain: `titles → titleauthor → authors` and `titles → publishers`.

In [ ]:
SELECT titles.title_id, titles.title, titles.type, publishers.pub_name, authors.au_fname, authors.au_lname
FROM titles
INNER JOIN titleauthor ON titles.title_id = titleauthor.title_id
INNER JOIN authors     ON titleauthor.au_id = authors.au_id
INNER JOIN publishers  ON titles.pub_id = publishers.pub_id

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>title_id</th><th>title</th><th>type</th><th>pub_name</th><th>au_fname</th><th>au_lname</th></tr></thead><tbody><tr><td>PS3333</td><td>Prolonged Data Deprivation: Four Case Studies</td><td>psychology</td><td>New Moon Books</td><td>Johnson</td><td>White</td></tr><tr><td>BU1032</td><td>The Busy Executive&#x27;s Database Guide</td><td>business</td><td>Algodata Infosystems</td><td>Marjorie</td><td>Green</td></tr><tr><td>BU2075</td><td>You Can Combat Computer Stress!</td><td>business</td><td>New Moon Books</td><td>Marjorie</td><td>Green</td></tr><tr><td>PC1035</td><td>But Is It User Friendly?</td><td>popular_comp</td><td>Algodata Infosystems</td><td>Cheryl</td><td>Carson</td></tr><tr><td>BU1111</td><td>Cooking with Computers: Surreptitious Balance Sheets</td><td>business</td><td>Algodata Infosystems</td><td>Michael</td><td>O&#x27;Leary</td></tr><tr><td>TC7777</td><td>Sushi, Anyone?</td><td>trad_cook</td><td>Binnet &amp; Hardley</td><td>Michael</td><td>O&#x27;Leary</td></tr><tr><td>BU7832</td><td>Straight Talk About Computers</td><td>business</td><td>Algodata Infosystems</td><td>Dean</td><td>Straight</td></tr><tr><td>BU1032</td><td>The Busy Executive&#x27;s Database Guide</td><td>business</td><td>Algodata Infosystems</td><td>Abraham</td><td>Bennet</td></tr><tr><td>PC8888</td><td>Secrets of Silicon Valley</td><td>popular_comp</td><td>Algodata Infosystems</td><td>Ann</td><td>Dull</td></tr><tr><td>TC7777</td><td>Sushi, Anyone?</td><td>trad_cook</td><td>Binnet &amp; Hardley</td><td>Burt</td><td>Gringlesby</td></tr><tr><td>PC9999</td><td>Net Etiquette</td><td>popular_comp</td><td>Algodata Infosystems</td><td>Charlene</td><td>Locksley</td></tr><tr><td>PS7777</td><td>Emotional Security: A New Algorithm</td><td>psychology</td><td>New Moon Books</td><td>Charlene</td><td>Locksley</td></tr><tr><td>TC4203</td><td>Fifty Years in Buckingham Palace Kitchens</td><td>trad_cook</td><td>Binnet &amp; Hardley</td><td>Reginald</td><td>Blotchet-Halls</td></tr><tr><td>TC7777</td><td>Sushi, Anyone?</td><td>trad_cook</td><td>Binnet &amp; Hardley</td><td>Akiko</td><td>Yokomoto</td></tr><tr><td>MC2222</td><td>Silicon Valley Gastronomic Treats</td><td>mod_cook</td><td>Binnet &amp; Hardley</td><td>Innes</td><td>del Castillo</td></tr><tr><td>MC3021</td><td>The Gourmet Microwave</td><td>mod_cook</td><td>Binnet &amp; Hardley</td><td>Michel</td><td>DeFrance</td></tr><tr><td>BU1111</td><td>Cooking with Computers: Surreptitious Balance Sheets</td><td>business</td><td>Algodata Infosystems</td><td>Stearns</td><td>MacFeather</td></tr><tr><td>PS1372</td><td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td><td>psychology</td><td>Binnet &amp; Hardley</td><td>Stearns</td><td>MacFeather</td></tr><tr><td>PS1372</td><td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td><td>psychology</td><td>Binnet &amp; Hardley</td><td>Livia</td><td>Karsen</td></tr><tr><td>TC3218</td><td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td><td>trad_cook</td><td>Binnet &amp; Hardley</td><td>Sylvia</td><td>Panteley</td></tr><tr><td>PC8888</td><td>Secrets of Silicon Valley</td><td>popular_comp</td><td>Algodata Infosystems</td><td>Sheryl</td><td>Hunter</td></tr><tr><td>MC3021</td><td>The Gourmet Microwave</td><td>mod_cook</td><td>Binnet &amp; Hardley</td><td>Anne</td><td>Ringer</td></tr><tr><td>PS2091</td><td>Is Anger the Enemy?</td><td>psychology</td><td>New Moon Books</td><td>Anne</td><td>Ringer</td></tr><tr><td>PS2091</td><td>Is Anger the Enemy?</td><td>psychology</td><td>New Moon Books</td><td>Albert</td><td>Ringer</td></tr><tr><td>PS2106</td><td>Life Without Fear</td><td>psychology</td><td>New Moon Books</td><td>Albert</td><td>Ringer</td></tr></tbody></table>

### Top 2 publishers by total quantity sold

In [ ]:
SELECT TOP (2) WITH TIES
    publishers.pub_id,
    publishers.pub_name,
    SUM(sales.qty) AS TotalQuantity
FROM publishers
INNER JOIN titles ON publishers.pub_id = titles.pub_id
INNER JOIN sales  ON titles.title_id = sales.title_id
GROUP BY publishers.pub_id, publishers.pub_name
ORDER BY TotalQuantity DESC;

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>pub_id</th><th>pub_name</th><th>TotalQuantity</th></tr></thead><tbody><tr><td>0736</td><td>New Moon Books</td><td style="text-align:right">208</td></tr><tr><td>0877</td><td>Binnet &amp; Hardley</td><td style="text-align:right">150</td></tr></tbody></table>

<a id="s10"></a>
## 10. Join practice: Northwind orders

### Exercise 10.1

*For every order show: order number, customer name, employee name, order date and the total value of the order.*

Tables: `Orders` (the center), `Customers`, `Employees`, `[Order Details]`. Square brackets are needed because the table name contains a space. Note two things:
* the employee name is built with the `+` operator (`FirstName + ' ' + LastName`), and because it is not an aggregate, the **same expression** must appear in `GROUP BY`;
* `Quantity * UnitPrice` is calculated per order line and then summed per order.

In [ ]:
use Northwind
go
SELECT
    Orders.OrderID,
    Orders.OrderDate,
    Customers.CompanyName AS CustomerName,
    Employees.FirstName + ' ' + Employees.LastName AS EmployeeName,
    SUM([Order Details].Quantity * [Order Details].UnitPrice) AS TotalOrders
FROM Orders
INNER JOIN Customers       ON Orders.CustomerID = Customers.CustomerID
INNER JOIN Employees       ON Orders.EmployeeID = Employees.EmployeeID
INNER JOIN [Order Details] ON Orders.OrderID    = [Order Details].OrderID
GROUP BY
    Orders.OrderID,
    Orders.OrderDate,
    Customers.CompanyName,
    Employees.FirstName + ' ' + Employees.LastName
ORDER BY TotalOrders DESC

<table border="1" class="dataframe"><thead><tr style="text-align: left;"><th>OrderID</th><th>OrderDate</th><th>CustomerName</th><th>EmployeeName</th><th>TotalOrders</th></tr></thead><tbody><tr><td style="text-align:right">10643</td><td>1997-08-25</td><td>Alfreds Futterkiste</td><td>Michael Suyama</td><td style="text-align:right">1086.00</td></tr><tr><td style="text-align:right">11011</td><td>1998-04-09</td><td>Alfreds Futterkiste</td><td>Janet Leverling</td><td style="text-align:right">960.00</td></tr><tr><td style="text-align:right">10692</td><td>1997-10-03</td><td>Alfreds Futterkiste</td><td>Margaret Peacock</td><td style="text-align:right">878.00</td></tr><tr><td style="text-align:right">10835</td><td>1998-01-15</td><td>Alfreds Futterkiste</td><td>Nancy Davolio</td><td style="text-align:right">851.00</td></tr><tr><td style="text-align:right">10952</td><td>1998-03-16</td><td>Alfreds Futterkiste</td><td>Nancy Davolio</td><td style="text-align:right">491.20</td></tr><tr><td style="text-align:right">10702</td><td>1997-10-13</td><td>Alfreds Futterkiste</td><td>Margaret Peacock</td><td style="text-align:right">330.00</td></tr></tbody></table>

### Try it yourself (no answers given)

1. In `pubs`, show the average price per publisher name (join `titles` and `publishers`) and keep only publishers whose average is above 12.
2. In `pubs`, count how many titles each author wrote, showing the author's full name with `CONCAT_WS`.
3. In `Northwind`, count orders per employee and show only employees with more than 1 order.
4. Use `GROUPING SETS` to show customers per country, plus the grand total, but no city rows.

<a id="s11"></a>
## 11. Cheat sheet

### Aggregates and `NULL`

| Expression | Result on `price` (18 rows, 2 `NULL`s) |
|---|---|
| `COUNT(*)`, `COUNT(1)`, `COUNT(title_id)` | 18 |
| `COUNT(price)` | 16 |
| `SUM(price)` / `SUM(ISNULL(price,0))` | 236.26 / 236.26 |
| `AVG(price)` | 14.7662 (÷ 16) |
| `AVG(ISNULL(price,0))` | 13.1255 (÷ 18) |

### Which clause for which job?

| I want to... | Use |
|---|---|
| keep only some **rows** | `WHERE` |
| keep only some **groups** (condition on an aggregate) | `HAVING` |
| a result **per category** | `GROUP BY` |
| unique values, no aggregate | `DISTINCT` |
| subtotals, hierarchical | `GROUP BY ... WITH ROLLUP` |
| subtotals, every combination | `GROUP BY ... WITH CUBE` |
| subtotals, exactly the ones I list | `GROUP BY GROUPING SETS (...)` |
| count rows meeting a condition | `COUNT(CASE WHEN ... THEN 1 END)` |
| rows matching in two tables | `INNER JOIN ... ON` |
| top N rows including ties | `TOP N WITH TIES` + `ORDER BY` |

### Common errors and their cause

| Error | Cause | Fix |
|---|---|---|
| Msg 8120 *not contained in either an aggregate function or the GROUP BY* | a plain column next to an aggregate | add it to `GROUP BY` or aggregate it |
| Msg 147 *An aggregate may not appear in the WHERE clause* | aggregate used as a row filter | move the condition to `HAVING` |
| Msg 207 *Invalid column name* (alias in `WHERE`) | `WHERE` runs before `SELECT` creates the alias | repeat the expression, or use a subquery/CTE |
| Msg 209 *Ambiguous column name* | same column name in two joined tables | prefix with the table alias |

### Logical processing order

`FROM` → `JOIN` / `ON` → `WHERE` → `GROUP BY` → `WITH CUBE/ROLLUP` → `HAVING` → `SELECT` → `DISTINCT` → `ORDER BY` → `TOP`